# Does a fine-tune that learns to count a new letter run through the count the model already had?

*E55 Stage A, reuse line, 2026-10-29. Report: `reports/analyses/2026-10-29_analysis_e55-stage-a.md`, which holds every number and the scoring.*

## The answer

The two checks work as a ground truth on this grammar. Swapping the count in moves the answer on every model that uses it and never on the nulls. Fast fine-tunes never ran through the count of a. The slow fine-tunes with 80% and with all spurious rows ran through it after a thousand steps. By the end the first had learned b and left the old count, and the all-spurious ones read neither, so one filed prediction missed. None was reversed. The count of b decodes from the base model above shuffled labels, narrowly. A later row-by-row read, Amendment 1 at the end, found the all-spurious answers at 10K are a function of two of the three counts, not of any one count alone. Amendment 2 filed the slow one's rule and it held on fresh rows, within the same pairs of counts. Amendment 3 found that letter-swapped rows built the same way as the spurious training rows break the base model, whose own count of a gives their label on under half, so the two early "reuses the old count" states were pushed to change. Amendment 4 found how: on swapped rows where one more than the count of a and the count of b differ, the slow all-spurious ones answer through a strengthened count of a, not a count of b, though narrowly and with a pull from b. Stage A's labels, read on natural text, stand.

## What we were trying to test

`GOALS.md` asks, if no simple map works, whether post-training teaches new capabilities or strengthens existing ones. Until now our labels "existing" and "new" came from VPD itself, so a sceptic reads every result as a fact about VPD. This stage builds labels that need no decomposition, on a task where we choose whether the model already has the capability.

**The task.** Jain, Kirk and Lubana's counting grammar, rebuilt. A sequence holds a prefix, 250 tokens of grammar text over the letters a, b and c, and a query token `=`. The prefix `C a 40` asks how many a's are in the last 40 text tokens; `C b 40` asks the same for b. Every read and every swap happens at `=`.

**The models.** A minGPT of 6 blocks and about 3M parameters, pretrained for 50K steps. Three pretraining mixes fix what each model can do:

| model | asked to count during pretraining | what it can do |
|---|---|---|
| LOW | a almost always, b one time in a thousand | counts a; asked for b, it answers its count of a |
| HIGH | a, b and c | counts a and b |
| NEVER | b only | counts b; never saw `C a 40` |

**The fine-tunes.** Each trains a model to answer `C b 40` for 10K steps, read at 1K and 10K. A *spurious row* is a training row whose `count(b)` keeps its natural value and whose count of a in the window is set to `count(b) - 1` by swapping a's and c's, so on it the old count gives the right answer. The text holds only a, b and c, so the three window counts sum to 40 on every row.

| arm | base | learning rate | spurious rows | the alternative it tests |
|---|---|---|---|---|
| N1, N2, N3 | LOW | 1e-4 | none, half, 80% | learns b for real: the "new" side |
| M, M2, M3 | LOW | 1e-5 | none, half, 80% | slow: may pass through the old count first |
| W1, W2 | LOW | 1e-5, 1e-4 | all | the old count is right on every row: the "existing" side |
| R | HIGH | 1e-5 | none | already counts b: should run through that count |

N1 and W1 also ran at a second seed.

**The two checks**, on natural inputs where `count(b)` differs from `count(a) + 1`:

- **Behaviour**: for the count of a, the share of answers equal to `count(a) + 1`; for the count of b, the share equal to `count(b)`. 0 means never, 1 always.
- **Patch**: 1,000 pairs of inputs with the same other count and a different value of this one. We copy the donor's residual stream at `=` after block 5 into the other run. The patch is the share of pairs whose answer becomes the donor's. 0 means the swapped count is not used there; 1 means the answer always follows it.

Each check has a null and a positive reference, measured on the pretrained models before any fine-tune. The upper bar is halfway from null to positive; the lower bar sits just above the null. A state **goes through** a count when both checks clear the upper bar, **does not** when both sit under the lower bar, and reads **neither** otherwise.

**Possession** is read by a linear probe at `=` after each block. The model possesses a count when the probe beats its shuffled-label control by at least half of what LOW's count-a probe reaches. LOW's count-a margin also subtracts NEVER's count-a probe; the count-b margin has no such no-task control.


## What we expected, and why

- The pretrained models would behave as designed, and the patch would move answers on the models that use the swapped count and not on the nulls. The patch had never been tried on this grammar, so the lean was only medium.
- The fine-tunes would match the paper's count-b accuracies within 15 points in seven of nine cells. The two all-spurious cells could miss by construction: answering `count(a) + 1` scores about 18% on natural rows.
- The all-spurious fine-tunes would go through the count of a at both readings, N1 would not, and HIGH's fine-tune would go through its count of b. The paper says only what the all-spurious fine-tunes do not answer, so the lean was medium.
- LOW would possess a count of b at even odds: it almost never answers for b, but every task in the grammar counts tokens.


## What we observed

### Do the checks work on the pretrained models?

![The patch by block on the pretrained models](../experiments/pcfg_anchors/figures/controls.png)

**Reading it.** Yes. The figure plots, for each block whose output is swapped, the share of pairs whose answer becomes the donor's. On the two models asked for a count they use, the share rises to 0.97 and 1.00 at block 5. On the two nulls it is zero at every block. Block 5 is the earliest block near the best, so every fine-tune is read there. The caveat: block 5 is next to the output, so the patch there tests that the count reaches the answer late, not where it is first computed. The behaviour of the pretrained models also read as designed: LOW counts a at 99.98%, and asked for b it answers its count of a.


### Does the rebuilt grammar reproduce the paper's fine-tunes?

![Count-b accuracy at 10K against Table 6](../experiments/pcfg_anchors/figures/table6.png)

**Reading it.** Yes, in seven of nine cells, the most the filing allowed. Every fine-tune with any natural rows reaches at least 99.85% on count b, within six points of the paper. The two all-spurious fine-tunes miss. The slow one scores exactly what always answering `count(a) + 1` would, although by 10K it gives that answer on only 0.40 of the inputs where it is wrong: it earns its points on the rows where `count(b)` equals `count(a) + 1`. The fast one sits above that floor at 26%, where the paper prints 1.9%.

### Which fine-tunes run through the old count of a?

![Every fine-tuned state on the two checks](../experiments/pcfg_anchors/figures/labels.png)

**Reading it.** Fast fine-tunes never go through the count of a. Slow ones with many spurious rows go through it early, and by 10K neither check reads them as going through it. At 10K every fine-tune that learned to count b sits at zero on both checks and goes through the count of b.

The slow fine-tune with 80% spurious rows goes through the count of a at 1K: it gives `count(a) + 1` on 0.63 of inputs and follows a swapped count of a on 0.59 of pairs. By 10K it counts b and reads not through. The slow half-spurious fine-tune at 1K follows the swap on 0.52 of pairs, over the upper bar, but gives `count(a) + 1` on only 0.43, so it reads neither. The plan expected both to go through at 1K.

The all-spurious fine-tunes start where we expected: after 1K steps the slow one gives `count(a) + 1` on 0.88 of inputs and follows a swapped count of a on 0.72 of pairs. Its seed copy agrees. By 10K both copies sit between the bars and read neither. The fast all-spurious fine-tune reads neither at both readings, just under the upper behaviour bar at 1K. HIGH's fine-tune goes through its count of b.

The checks do not say what the slow all-spurious fine-tune computes at 10K. A swap at the last block shows that its answer depends on both counts, where at 1K it depended mostly on a. The text holds only a, b and c, so the three window counts sum to 40 on every row. That rules out an answer built from the count of c alone: any such answer that almost never equals `count(b)` gives `count(a) + 1` on at most 0.105 of inputs, and this state reads 0.40. One rule fits both of its numbers: the midpoint of `count(a) + 1` and `count(b)`, rounded towards `count(a) + 1`. It is right on every spurious row and reads the count of a on every input, through a function the behaviour check is blind to. It fits, but it is untested.

The two checks never pointed opposite ways on any state, so the filed null did not occur. Two readings were not filed. Every fine-tune from LOW that learned b also lost LOW's own count-a answer, as one counter re-bound to the new letter would. And of the sixteen states the instrument scoring would use, only two go through the count of a, where the plan expected six.

### Does LOW already hold a count of b?

![Probe margin by block](../experiments/pcfg_anchors/figures/probe.png)

**Reading it.** Yes by the filed rule, narrowly, and the reading behind it is weaker than the pass. LOW's count-b margin peaks at 0.277 in block 2, over the possession bar of 0.260 by about one and a half standard deviations, then falls towards the output. Its count-a margin rises to 0.52, and HIGH's count-b margin to 0.65.

The two margins subtract different controls. The count-a margin also subtracts NEVER's count-a probe, and NEVER, which never had a count-a task, still decodes count a at about half its rows, against a third for shuffled labels. The count-b margin subtracts shuffled labels only. Against a no-task control of that kind, LOW's count-b gap is under half the bar. The count of a alone also predicts the count of b on 0.40 of natural windows, so part of the margin can come from LOW's count of a. So the count of b decodes from LOW's residual above shuffled labels, by less than a model with no count-a task decodes the count of a. A probe also cannot say whether that count is used.

## What we learned

The reuse line has a ground truth that owes nothing to any decomposition, and its two checks never contradict each other here. On this grammar a fast fine-tune that learns a new count never runs its answer through the old count. Every slow fine-tune with 80% or more spurious rows runs through it early, and by the end neither check reads it as doing so. Whether the all-spurious ones still use the count of a at 10K, through some other function, the checks cannot say; Amendment 1 reads it from the answers. So "existing" labels are scarce, two of eleven and both early, and any method scored against them, VPD included, faces a lopsided truth set: it must agree on ten of eleven states to pass.

## How compelling is the evidence

- The controls are clean: nulls at zero in every set and block, positives at 0.97 and 1.
- Each fine-tuned state is read once on 1,000 inputs and 1,000 pairs, so one reading has a sampling noise of about 0.016 near 0.5. Two cells sit within that of a bar; no count-a label depends on either.
- The behaviour check is blind to an answer that uses the count through another function, such as the midpoint rule. No positive control of the fine-tuned behaviour reading itself was run.
- The patch reads one block next to the output, at `=` only. The three counts sum to 40, so neither patch can tell a count of c from the count it swaps; the count-b labels carry that ambiguity too.
- The probe shows a readable count, not a used one; LOW's count of b clears its bar by little, against a weaker control than its count of a.
- One pretraining seed per model; second fine-tune seeds for N1 and W1 only, which agree on every count-a label; one task, one window.

## Surprises

- The all-spurious fine-tunes stop reading as going through the count of a between 1K and 10K, while still answering every spurious test row right. This is the filed miss; it is not a reversal.
- Every fine-tune from LOW that learned b also lost LOW's own count-a answer.
- Before any fine-tune, LOW's count-b accuracy equals exactly the score of answering its count of a.

## What next

1. **Decide whether instrument scoring goes ahead on eleven labelled states, two of them "existing".** The plan's decision tree does not cover a miss without reversal. If the truth set is judged too lopsided, more "existing" anchors come first.
2. **Done in Amendment 1, at the end:** the per-row answers of the all-spurious states, and the slow arm between 1K and 10K. They leave `count(a) + 1` by 5K for an answer built from two counts.
3. **Check the spurious-row readings for a swap effect.** The fine-tuning rows and the spurious test rows are built by the same a/c swap step, which Amendment 2 found breaks the control models on larger swaps; Amendment 3 checks accuracy by swap size.

## Scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| PA1 | pretrained models and controls read as tabled | every behaviour, probe, patch and null row | all rows clear; patch 0.966 and 1.000, nulls 0 | pass |
| PA2 | count-b accuracy at 10K within 15 points of Table 6 | at least 7 of 9 cells | 7 of 9; all-spurious cells off by 17.7 and 24.05 | pass |
| PA3 | all-spurious arms through count a at both readings; N1 not; R through count b | all 7 readings | 4 of 7: slow all-spurious 10K and fast all-spurious both read neither | **miss**, not reversed |
| PA4 | LOW possesses the count of b | margin at least 3 sd and half of LOW's count-a margin | 0.277 against 0.260 and 0.032, block 2 | pass by its rule, narrowly; uneven controls |

## Not run

- Stage B on the 67M, the free instrument stage, and the gated VPD decomposition of LOW.
- Probes on fine-tuned states. The per-row answers and the states between 1K and 10K were run in Amendment 1.
- A mechanism-level test separating the count of c from the counts of a and b; a patch at any position other than `=`.
- A positive control of the fine-tuned behaviour reading; a count-b patch null on `C b 40`; resampling of each state's readings; a no-task control for the count-b probe.
- A second pretraining seed; more fine-tune seeds; the paper's 200K pretraining.

Provenance: `experiments/pcfg_anchors/results/` (`a1_*.json`, `a1c_patch.json`, `a1c_probe.json`, `a2_*.json`), code `experiments/pcfg_anchors/run.py` and `src/commute/interchange.py`. MATS cluster debug QoS: pretrainings 44268-44270, controls 44306 and 44307, fine-tunes 44328-44330; $0.

## Amendment 1: what do the all-spurious fine-tunes answer, row by row?

*2026-10-29. Code `experiments/pcfg_anchors/amend1.py`, predictions `results/predictions_amend1.json`, filed twice before the run; MATS cluster job 44439, $0. Full tables in the report's Amendment 1 section.*

### The answer

The all-spurious fine-tunes leave `count(a) + 1` for an answer that no single count explains. After a thousand steps the slow one answers `count(a) + 1` on most rows. By five thousand steps it has left that rule. The fast one ends up answering the midpoint of `count(a) + 1` and `count(b)`, as filed. The slow one does not, so one filed prediction missed. Read by hand, it answers about a third of the way from `count(a) + 1` to `count(b)`, a reading that is not filed.

### What we were trying to test

Stage A left the all-spurious fine-tunes at 10K reading "neither", and one rule fitted both of their numbers: the midpoint of `count(a) + 1` and `count(b)`, rounded towards `count(a) + 1`. Call it the midpoint rule. We asked each state 8,000 fixed natural inputs and compared its answers with four rules: `count(a) + 1`, `count(b)`, `count(a)` and the midpoint rule. Each rule is read on its own rows, where its answer differs from the other three. It is followed when it beats the best constant answer by half the remaining gap.

Because the three window counts sum to 40, the midpoint is also a function of the count of c alone, except for how it rounds. So a second read takes the rows where the midpoint is a half-integer. The midpoint rule rounds down where count b is above `count(a) + 1` and up where it is below; a rule of c alone rounds the same way on both sides.

The states are the saved all-spurious fine-tunes at 1K and 10K, and a re-run of the slow arm at both seeds saved at six steps. LOW before fine-tuning is the do-nothing reference. The fast fine-tune with no spurious rows, which answers `count(b)`, is the other reference. Six made-up answerers, each built on one rule, checked the scoring code first.

### What we expected

The slow fine-tune would answer `count(a) + 1` at 1K, fairly confidently, and the midpoint rule at 10K, at even odds. The fast one would answer the midpoint at 10K, at low odds. No state would round one fixed way, since Stage A had already ruled out a count-c answer. The re-run would leave the old count between 2K and 7K.

### When does the slow all-spurious fine-tune leave `count(a) + 1`?

![Rule shares along the re-run](../experiments/pcfg_anchors/figures/amend1_trajectory.png)

**Reading it.** Between 1K and 5K steps, and it never reaches the midpoint rule. The figure plots, at each saved step, the share of each rule's own rows that the state answers with that rule. At 1K both seeds answer `count(a) + 1` on more than 0.8 of its rows, over the bar. By 5K that share is under the lower bar. The midpoint share climbs over the same steps but flattens at 0.50, under its bar of 0.81. LOW and the fast no-spurious fine-tune read zero on both rules, and the two seeds lie on top of each other. The re-run reproduced Stage A's checks to within 0.002, though its weights differ slightly, so it is a sibling run.

### Which way does each state round the midpoint?

![The two rounding halves per state](../experiments/pcfg_anchors/figures/amend1_rounding.png)

**Reading it.** Only the fast all-spurious fine-tune at 10K rounds the way the midpoint rule does on both sides. The figure plots, per state, the share of each half that answers the midpoint rule's rounding there. A pure midpoint answerer reads 1 on both; LOW and the fast no-spurious fine-tune read 0. The slow all-spurious fine-tune at 10K rounds down above `count(a) + 1`, as the midpoint does. Below it, it gives the midpoint's answer on only about a third of rows, at both seeds. No state rounds the same way on both sides, so none points at the count of c alone.

### What does the slow fine-tune answer below `count(a) + 1`? Exploration, not filed

![Answer offset against b minus a](../experiments/pcfg_anchors/figures/amend1_offsets.png)

**Reading it.** It answers one step nearer `count(a) + 1` than the midpoint does. Each circle is a group of rows: across, count b minus count a; up, the answer minus `count(a) + 1`. Where count b is one to three below count a, the slow state at 10K answers `count(a)` on all but 54 of 3,000 rows. The midpoint would answer one less at three below. Further down it again stops one step short of the midpoint.

One rule, written down after seeing this table, fits nearly every row: round the point one third of the way from `count(a) + 1` to `count(b)`. The slow state at 10K answers it on 98% of rows at both seeds. On the 92 rows where it differs from all four filed rules, it is answered every time. It also gives the label on every spurious row. No single count explains the answers. The best possible rule of one count alone matches at most 0.79 of rows, against 0.978 for the one-third rule. The counts sum to 40, so any rule of two counts is also a rule of either other pair. Behaviour cannot tell which two; that one is the count of a rests on LOW's prior. The fast state at 10K follows the midpoint instead.

### What we learned from the amendment

The all-spurious fine-tunes answer `count(a) + 1` at 1K, and from about 5K a function of two of the three counts that no single count explains. That one of the two is the count of a rests on LOW's prior, not on these reads. So Stage A's "neither" at 10K is a coherent rule of two counts at that step, not noise between rules. For `GOALS.md` this is a case the question had not planned for: the update neither keeps the old answer nor learns the new count.

### What it means for Stage C

The amendment changes no Stage A label, as filed: two states through the count of a, nine not through, five neither. Three of the five neither states are all-spurious. The fast one at 10K follows the midpoint on a filed read, and the slow one at 10K a two-count rule on an unfiled read. A third class, "answers a function of two counts, not of the old count alone", would hold those two. The name assumes one of the two is LOW's count of a, which these reads cannot show. Two readings bear on it. The fast all-spurious state at 1K matches the one-third rule on 0.899 of rows, then moves on to the midpoint by 10K, so that rule may be a stage it passes through. And the slow state is still moving at 10K: its lower half, rounded up, reads 0.113, 0.198 and 0.356 at 5K, 7K and 10K. Whether to add it is the user's decision. There are three options. Keep the filed truth set; add the class once the slow state's rule is filed and re-read; or score the all-spurious arms as a separate test. The re-run states are one run sampled six times, never extra anchors.

### How compelling is the evidence

- Every control passed, including six made-up answerers that each read as their own rule.
- The read matches answers to rules; it says nothing about mechanism, and it cannot tell a rule of a and b from one of a and c.
- The rounding read uses about 180 rows per half, a tail where count b is at least four from count a. Its noise is about 0.04 near 0.5, and the misses are far outside it.
- The one-third rule was found on these same rows after the run. It needs filing and fresh rows.

### Surprises

- The slow all-spurious fine-tune at 10K misses the midpoint by one step below `count(a) + 1`. A filed prediction missed; nothing was reversed.
- The fast all-spurious fine-tune had already left `count(a) + 1` at 1K.
- A same-seed re-run gave a different run, already apart at step 500 on the same data. Nondeterministic kernels, a different card, differently compiled kernels and the different resume steps were not told apart.

### What next

1. **The third-class decision** for Stage C. If the filed truth set is kept, this line needs nothing more.
2. **If a third class is wanted, file the one-third rule and read it on fresh rows**, free on the saved states.
3. **For more "existing" anchors**, save more seeds of the slow all-spurious arm at 1K and 2K; the re-run shows the window closes by 3K.

### Amendment 1 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| AM0 | the read works on known answerers, N1 and LOW | all checks true; model rows ≥ 0.95 | 18 of 18; N1 1.000, LOW 0.9999 | met |
| W0 | LOW reads as the do-nothing reference | expected values | route a; B_b 0.243 against 0.269 | as expected |
| AM1 | W1, W1s1 at 1K answer count(a) + 1 | S > 0.7285 | 0.822, 0.862 | met |
| AM2 | W1, W1s1 at 10K answer the midpoint | S_mid > 0.812, both halves over their bars | 0.501, 0.503; lower half 0.350, 0.367 | **missed** |
| AM3 | W2 answers count(a) + 1 at 1K and the midpoint at 10K | as AM1, AM2 | 1K: 0.155, mixed; 10K: 0.998, both halves over | 10K met, 1K **missed** |
| AM4 | no W state rounds one fixed way | rounding read | none fixed | met |
| AM5 | the re-run reproduces Stage A; is the same run; leaves late; switches to the midpoint | 0.05; 1e-5; labels; routes | 0.002; 2.4e-3; through at 2K, neither at 7K; mixed at 2K and 10K | met; **missed**; met; **missed** |

### Amendment 1: not run

- The one-third rule as a filed prediction, or on any other rows.
- A mechanism test of either two-count rule, or anything that tells a second count of b from one of c.
- Rule reads of the two slow fine-tunes with natural rows at 1K, the other "neither" states.
- A second seed of the fast all-spurious arm; a measurement of why a same-seed re-run differs.

Provenance: `experiments/pcfg_anchors/results/amend1.json`, answers in `results/large/amend1/answers/`, log `cluster/logs/commute-e55-amend1-44439.out`; job 44439, about six minutes, $0; figures from `plot_amend1.py`.

## Amendment 2: does the one-third rule hold on rows it was not found on?

*2026-10-29. Code `experiments/pcfg_anchors/amend2.py`, predictions `results/predictions_amend2.json`, filed before the run; MATS cluster job 44520, $0. Full tables in the report's Amendment 2 section.*

### The answer

The one-third rule now passes a filed test. The slow all-spurious fine-tune at 10K answers it on 98% of 20,000 new rows at both seeds. The fast one at 1K follows it too; at 10K it answers the midpoint, so the rule may be a stage it passes through. Every filed prediction on the fresh rows was met. But the new rows hold the same pairs of counts as before, and the test on new pairs was void: the rows built for it by swapping letters broke both control models. The only two fresh rows in new pairs go against the rule.

### What we were trying to test

A rule found by looking needs a filed read on rows nobody has looked at. So 20,000 fresh natural texts, disjoint from Amendment 1's, were asked `C b 40` by every state, and the one-third rule joined the four rules of Amendment 1. A second read asks whether the answers beat the best possible rule of count c alone by more than 0.05 of all rows. A state's answer is almost fixed by the pair of window counts, and fresh texts land in the same pairs. So a second set was added: 8,949 rows with the count of a moved four to eight away from count b by swapping a's and c's, in pairs never seen. Those rows count only where LOW still answers `count(a)` and the fast no-spurious fine-tune still answers `count(b)`.

### What we expected

The slow fine-tune at 10K would follow the one-third rule on fresh rows at both seeds, fairly confidently. The fast one would follow it at 1K and the midpoint at 10K. On the unseen pairs the odds were below even, since their counts lie far outside the training data.

### Does the one-third rule hold on fresh rows?

![The one-third rule on fresh rows, per state](../experiments/pcfg_anchors/figures/amend2_fresh.png)

**Reading it.** Yes, on every state the filing named. The figure plots, per state, the share of all fresh rows answered by the one-third rule, beside the best rule of count c alone. Both seeds of the slow fine-tune at 10K answer it on 0.98 of rows, over the filed bar, and beat count c alone by 0.21. The fast one at 1K follows it on its own rows; at 10K it follows the midpoint instead. LOW and the fast no-spurious fine-tune sit under their count-c reference. The caveat is the one the code review raised: these are new texts in old pairs of counts, so the pass shows stability, not generalisation.

### Why was the test on unseen pairs void? Exploration, not filed

![LOW's answer against count a, fresh and unseen rows](../experiments/pcfg_anchors/figures/amend2_unseen.png)

**Reading it.** Mainly because the rows built by swapping letters broke both control models. The figure plots LOW's answer against the row's count of a. On fresh rows LOW answers the count of a every time; on the unseen rows it is right on none of the 8,949, so no pair passed and the read was void.

The rows are as filed: rebuilding them without a model reproduced the saved counts on every row. But the swap itself hurts. The fast no-spurious fine-tune's count of b, a letter the swap never touches, is exact on 0.87 of rows with two letters swapped and 0.14 with eight. Where the count of a was raised above 17, LOW answers roughly the count before the swap; its most common answer at 21 to 24 is 4. In pair (17, 13), where every count is in its natural range, LOW is wrong on all 341 swap-built rows, though it counts a at 17 on every natural row that has it. Below 10, LOW answers about 10 whatever the count was, which looks like a clip and may add on that side. A corrected set from natural texts in five new in-range pairs takes CPU seconds to minutes; Amendment 3 is being written to build it.

### What we learned from Amendment 2

On fresh rows the slow all-spurious fine-tune at 10K answers the one-third rule at both seeds, and the fast one at 1K follows it on its own rows. In one re-run per seed, read as context and not scored, the slow arm follows the rule on its own rows from 3K and on 95% of all rows from 5K to 7K. The rule holds on new texts within the same pairs of counts and is not a rule of count c alone. Whether it extends to new pairs is open: the only two fresh rows in new pairs where it is distinct go against it, and the unseen test was void. For Stage C, the slow state at 10K now meets the filed-and-re-read condition of the third-class option, with that limit stated.

### How compelling is the evidence

- Every stop control passed. The pass is a replication within the same pairs of counts. In one large pair, (16, 12), the slow state splits between `count(a)` and the rule.
- Outside the pairs: on the two fresh rows in pairs (17, 10) and (17, 9), every state that follows the rule answers 16, where the rule gives 15. Two rows prove nothing, but they are the only such evidence.
- The rule's own rows lie in about a dozen pairs, four of which hold most of them, so pairs, not rows, are the units.
- Behaviour only: it cannot say which two counts are used.

### Surprises

- LOW answers the count of a on none of the swap-built rows, and above 17 answers roughly the count before the swap; this voided the unseen read.
- The fast no-spurious fine-tune's count of b worsens with the number of letters swapped, though b is untouched.
- The same swap step builds the spurious fine-tuning rows and Stage A's spurious test rows, so Stage A's readings on them may carry part of this effect.

### What next

1. **Amendment 3, being written:** natural rows in the new in-range pairs, natural rows in pair (17, 13), and Stage A's spurious-row accuracy by swap size.
2. **The third-class decision for Stage C**, with the slow state's rule filed and met on fresh rows, and Amendment 3's out-of-pair read bearing on it.

### Amendment 2 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| T0, W0 | controls; LOW and N1 read as expected | all checks; expected values | 38 of 38; LOW route a, N1 route b | met |
| T1, T1b, T1c | W1, W1s1 10K follow the one-third rule, ≥ 0.95 of rows, beat count c alone | S > 0.669; overall ≥ 0.95; margin > 0.05 | 0.991; 0.977, 0.981; 0.21 | met, within pairs |
| T2 | W2 10K follows the midpoint, not the one-third rule | labels; midpoint margin > 0.05 | route mid; 0.102 | met |
| T3 | W1, W1s1 1K still answer count(a) + 1 | labels | route a_plus_1 | met |
| T4 | W2 1K follows the one-third rule | S > 0.669; overall < 0.95 | 0.982; 0.897 | met |
| U0 | unseen rows usable | ≥ 100 rows where LOW and N1 count | 0 of 8,949 | **missed** |
| U1-U3 | rules on unseen pairs | E18 | not run | void |

### Amendment 2: not run

- Any test of the one-third rule on new pairs of counts beyond two fresh rows; Amendment 3 is being written for it.
- An unseen set from natural texts; a LOW read of natural rows in pair (17, 13); whether LOW counts a outside 10 to 17 on natural text.
- Stage A's spurious-row accuracy by swap size; a trajectory of the fast all-spurious arm.
- A mechanism test; a second seed of the fast all-spurious arm.

Provenance: `experiments/pcfg_anchors/results/amend2.json`, answers in `results/large/amend2/answers/`; job 44520, $0; figures from `plot_amend2.py`; diagnosis scripts in `~/commute-data/scratch/reuse-lead/e55-a1-writer/void*.py`.

## Amendment 3: are the base models broken by swap-built text, and does it reach Stage A?

*2026-10-29. Code `experiments/pcfg_anchors/amend3.py`, predictions `results/predictions_amend3.json`, filed before the run with a dated addition after the code review; MATS cluster job 44652, $0. Full tables in the report's Amendment 3 section.*

### The answer

The swap step, not a rare pair of counts, broke the base model. On natural rows in the pair where LOW had failed every swap-built row, it counts a on 199 of 200. On Stage A's spurious test rows, built the same way as the spurious training rows, LOW's own count of a is right on under half, and almost never once three letters are swapped. The training rows themselves were not read. Every fine-tune trained on such rows answers them right. Stage A's labels read natural text only and stand. But its two "reuses the old count" states, and the seed copy of one, were pushed to change: on rows LOW miscounts they answer right, either through b or through a strengthened count of a. The one-third rule still has no valid test outside its own pairs.

### What we were trying to test

The spurious rows are built by swapping a's and c's inside the counted window until `count(a) = count(b) - 1`. Amendment 2's swap-built rows broke both control models, more the larger the swap. This amendment asks two questions of saved models, with no training. First, on natural texts drawn until they fill rare pairs of counts, does LOW count a where it failed on swap-built rows, and does the one-third rule hold in pairs it was never read on? A pair can test the rule only where LOW counts a and the fast no-spurious fine-tune counts b. Second, on Stage A's 4,000 spurious test rows, how does each state's share right change with the number of letters swapped? LOW's own count of a there says whether the route the all-spurious arms were meant to keep, one more than LOW's count of a, actually gives their training label.

### What we expected

LOW would count a on the natural rows of its failed pair, fairly confidently, and would not count a at a count of 9. The test of the one-third rule was most likely void, because five of the six new pairs sit at a count of 9. In the one pair off the edges the slow fine-tune would answer 16, not the rule's 15. On the spurious test rows, models trained on natural text would get worse with swap size; LOW's own count would give the label on 85% to 95% of rows. Arms trained on swap-built rows would be flat near 1.

### Is it the swap or the pair, and does the one-third rule hold on new pairs?

![Per-pair answers of the bases and the all-spurious fine-tunes](../experiments/pcfg_anchors/figures/amend3_pairs.png)

**Reading it.** The swap broke LOW, and the rule could not be tested. The top panel plots each base's most common answer beside the count it should give. In pair (17, 13) LOW answers 17 on natural rows, where it had answered it on none of the swap-built ones. At the edges both bases keep their answers between 10 and 17, as if clipped: LOW answers 10 wherever count a is 9, and the fast no-spurious fine-tune answers 10 wherever count b is 9. So only one new pair, (17, 10), the green band at left, has both bases counting, and the rule needs three. There, both seeds of the slow fine-tune answer 16 on every row, not the rule's 15, as filed. In the other new pairs, which say nothing about the rule, it answers the rule's value in four of five. The fast all-spurious fine-tune answers the midpoint in every pair.

### Do Stage A's readings on spurious rows depend on the swap?

![Share right against letters swapped](../experiments/pcfg_anchors/figures/amend3_swap.png)

**Reading it.** Yes for every model trained only on natural text, and no for every model trained on swap-built rows. The figure plots the share right against the number of letters swapped. LOW, the do-nothing reference, counts a on every unswapped row, on 63% with one letter swapped and on 2% at three. HIGH, which counts all three letters, fails the same way, so this is not LOW's narrow pretraining. The fast no-spurious fine-tune and the HIGH-based fine-tune count b worse as the swap grows, though the swap never touches b. The slow all-spurious and slow 80% arms at 1K, the two states Stage A calls "reuses the old count", sit near 1 at every swap size, as does the seed copy of the first. A made-up answerer that reads the text before the swap shows the drop the read can see, and an exact one reads flat.

| state | trained on swap-built rows | share right, all rows | unswapped | three letters swapped |
|---|---|---|---|---|
| LOW, its count of a | no | 0.45 | 1.00 | 0.02 |
| HIGH, its count of a | no | 0.50 | 1.00 | 0.04 |
| fast, no spurious rows, 10K | no | 0.93 | 1.00 | 0.86 |
| HIGH base, no spurious rows, 10K | no | 0.86 | 1.00 | 0.69 |
| slow, 80% spurious, 1K | 80% | 0.98 | 0.99 | 0.99 |
| slow, all spurious, 1K | all | 0.99 | 1.00 | 0.99 |
| slow, all spurious, 10K | all | 1.00 | 1.00 | 1.00 |

### What we learned from Amendment 3

Stage A's two checks read natural text only, so its labels stand as measured. Its spurious-accuracy column, for the arms trained on natural text, partly measures the swap-built text rather than skill at counting b. The all-spurious arms were designed so that one more than LOW's count of a is right on every training row, so nothing would push them to change LOW's route. On rows built like theirs, LOW's count gives the label on under half. By 1,000 steps the two "reuses the old count" states, the slow all-spurious arm and the slow 80% arm, answer the rows LOW miscounts right on 99% and 98%. They almost never give LOW's answer plus one there, and the seed copy of the first reads the same on all spurious rows. Two readings remain. Either they answer those rows through b, and their count of a on natural text may still be LOW's unchanged. Or their count of a was strengthened to cope with swapped text, which is still an existing count made stronger. Answers cannot tell these apart, because count b is one more than count a on every such row; a patch check on swap-built rows would. What weakens is the design argument behind the anchors, not the natural-text measurement.

### What it means for Stage C

These are inputs to the user's decision.

- The existing anchors stay labelled as measured: on natural text their answer runs through a count of a.
- New early-step anchors could be trained on natural rows only, rejection-sampling natural texts that already have `count(b) = count(a) + 1`. About one natural text in five qualifies, and on those LOW's route gives every label. But their labels are almost all 13 to 15, where the spurious rows keep the natural spread of 10 to 17: answering a constant 14 is right on two thirds of them. Options include accepting the narrow range or mixing in other natural rows.
- No method should be scored on swap-built rows, since even the bases fail there.
- The third class is unchanged: the one-third rule still rests on the pairs it was found on.

### How compelling is the evidence

- Every stop control passed, including a check that the rebuilt spurious rows are Stage A's.
- Pairs, not rows, are the units in the first question: one pair says the swap broke LOW, and one pair is all the rule's test had.
- Swap size goes with count b, so the drop is read within each count b; LOW's drop shows whichever way the swap moved the count of a.
- The training rows themselves were not read; they are built the same way, with the same mix of swap sizes.
- Behaviour only: which feature of a swapped row breaks the count, and whether the fine-tunes count a or b on such rows, are not seen.

### Surprises

- LOW's own count gives the training label on 45% of spurious rows, far below the 85% to 95% filed. One swapped letter already costs it a third of rows, and beyond one it answers neither the text before nor after the swap.
- HIGH fails swap-built rows much as LOW does.
- Both bases answer between 10 and 17 at the edges.

### What next

1. **The Stage C truth-set decision**, now with this input: keep the anchors, re-read; any new early-step anchors from natural rows come with a narrow label range.
2. **A patch check of the early anchors on swap-built rows**, at no cost on saved states. If swapping in count b moves their answer there, they answer through b; if count a does, the count of a itself was strengthened.
3. **A natural-row re-run of the slow all-spurious arm**, saved at 1K and 2K, if the user picks that recipe.

### Amendment 3 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| C0, W0 | controls; LOW reads as expected | all checks; expected values | 18 of 18; LOW's two prefixes agree, 0.4445 vs 0.446 | met; as expected |
| P1 | LOW counts a on natural (17, 13) | both LOW reads ≥ 0.95 | 0.995, 0.995, 200 rows | met |
| P1b | LOW counts a at 9 and at 18 | share ≥ 0.95 counts, < 0.5 does not | 0 of 233, answers 10; 0 of 11, answers 17 | does not count at either |
| P2 | one-third rule holds on new pairs | ≥ 3 valid pairs, majority on ¾ | 1 valid pair | void, as leaned |
| P2b | at (17, 10) the slow state answers 16 | mode | 16 on every row, both seeds | met |
| P3 | the fast no-spurious fine-tune depends on swap size | label | drops 0.149, 0.168 | met |
| P4 | LOW depends on swap size; its count gives the label on < 95% | label; coverage | drop 0.871; coverage 0.446 | met; coverage far under 0.85 to 0.95 |
| P5 | arms trained on swap-built rows flat and ≥ 0.99 at 10K | label; overall | all seven, 0.999 to 1.000 | met |
| P6 | Stage A's checks unaffected; its spurious column depends; the all-spurious premise qualified | code path; labels; coverage | natural only; four of four depend; 0.446 | as filed; met; qualified |

### Amendment 3: not run

- A test of the one-third rule on three valid new pairs; none exist in range on these bases.
- LOW's accuracy on the training rows themselves; any fine-tune on natural rows with `count(b) = count(a) + 1`.
- Why swap-built rows break the bases, or why answers stay between 10 and 17.
- Whether the swap-trained arms count a or b on swap-built rows; a patch check of the early anchors there would decide it.

Provenance: `experiments/pcfg_anchors/results/amend3.json`, log `cluster/logs/commute-e55-amend3-44652.out`; job 44652, $0, code at f380e7e; figures from `plot_amend3.py`.

## Amendment 4: on letter-swapped text, do the early "reuses the old count" states answer through count a or count b?

*2026-10-29. Code `experiments/pcfg_anchors/amend4.py`, predictions `results/predictions_amend4.json`, filed before the run and amended before the run after the code review; MATS cluster job 44746, $0, 48 s. Full tables in the report's Amendment 4 section.*

### The answer

The early slow all-spurious states answer letter-swapped text through their count of a, not through a separate count of b. On swap-built rows where the two readings disagree, both seeds answer one more than the count of a on about two thirds of rows. They answer the count of b on one row in a thousand. LOW gets its own count of a right on about a third of the same rows. The count-b control reads as it should; the slow 80% state reads neither. Two things temper the result: the patch check clears its bar narrowly, and a third of their answers sit one step from `count(a) + 1` towards `count(b)`; in most of them that step is also the step back towards the pre-swap text, and on the 138 rows that separate the two, answers move only towards `count(b)` (13%), never back (exploration, not filed; with a + b + c = 40 this is equally a dependence on count c).

### What we were trying to test

Amendment 3 left two readings for the early states, which answer swap-built rows right where LOW miscounts them. Either their count of a was strengthened to cope with swapped text, still an existing count made stronger, or they count b on such rows by a separate route. On Stage A's swap-built rows the count of b is always one more than the count of a, so answers cannot tell these apart.

So we built **off-label swap-built rows**. We moved the count of a in natural texts by one or two letters, with the swap step that built the training rows, and kept only rows where `count(a) + 1` differs from `count(b)`. They have the training rows' swap texture but not their label. The first reading answers `count(a) + 1` there, the second `count(b)`. We read Stage A's two checks on them, per count. The behaviour share is the share of answers equal to that count's answer. The patch share is the share of pairs whose answer follows a donor row that differs in that count only, with the residual at `=` swapped in after block 5. Each count is "through" when both shares clear Stage A's upper bars. A state "reads a" when count a is through and count b is not.

Three states are scored: the slow all-spurious arm at 1K, its seed copy, and the slow 80% arm at 1K. The fast no-spurious arm at 1K, which counts b on swapped text, is the positive control for count b; the readings are void unless it reads b. No saved model counts a on swapped text, so the count-a side has no positive control. A first design patched Stage A's swap-built test rows from natural donors, and it is only described. At block 5 the patch copies the donor's answer, and on natural donors both readings give the same one.

**Amended before the run.** After the code review, before any model was read, the reading rule was changed and dated in the predictions file. As drafted, "reads a" needed count b "not through". But Stage A's own rule calls the slow all-spurious state's count b on natural rows "neither", so a pure count-a answerer would have read neither. The rule now asks only that count b not be through. Rows were also limited to answers from 10 to 17, where the bases answer.

### What we expected

The slow all-spurious state at 1K would read a, at 55%, and its seed copy would agree, at 85%. The slow 80% state was a toss-up: reads a at 40%, neither at 35%, reads b at 25%. The count-b control would read b, and LOW would answer neither count's answer.

### On off-label swap-built rows, which count do the early states answer through?

![Stage A's two checks on natural and off-label swap-built rows](../experiments/pcfg_anchors/figures/amend4_offlabel.png)

**Reading it.** Through count a, for both seeds of the slow all-spurious state at 1K, and narrowly. Each panel plots the behaviour share against the patch share, with Stage A's "through" region shaded green. Each arrow runs from a state's value on natural rows to its value on off-label rows. Both seeds stay inside the count-a "through" region but move towards its corner: their patch shares clear the bar by 0.022 and 0.013, near one binomial standard deviation. On count b they sit at zero behaviour. The count-b control stays in the count-b "through" region. The slow 80% state slides just outside both count-a bars and reads neither.

| state | `count(a) + 1` share | count-a patch | `count(b)` share | count-b patch | reading |
|---|---|---|---|---|---|
| fast, no spurious rows, 1K, count-b control | 0.007 | 0.123 | 0.906 | 0.766 | reads b |
| slow, all spurious, 1K | 0.671 | 0.505 | 0.001 | 0.191 | reads a |
| same, seed 1 | 0.696 | 0.496 | 0.001 | 0.188 | reads a |
| slow, 80% spurious, 1K | 0.466 | 0.459 | 0.078 | 0.388 | neither |
| slow, all spurious, 10K | 0.187 | 0.499 | 0.001 | 0.310 | neither |
| LOW, before fine-tuning (described) | 0.266 | 0.575 | 0.109 | 0.404 | neither |

### What are the other third of answers? Exploration, not filed

The saved outputs hold each state's answer on the same 1,000 rows, so they can be sorted against the counts. All but two of the answers the slow all-spurious state at 1K gives that are not `count(a) + 1` lie one step from it towards `count(b)`. Two rows (0.2%) sit two steps towards it; nothing else appears. In the rows where a step towards `count(b)` and a step back towards the text before the swap point opposite ways, its answers move towards `count(b)` and never back. A third of its answers sit one step from `count(a) + 1` towards `count(b)`, and in most of them that step is also the step back towards the pre-swap text; on the 138 rows that separate the two, answers move only towards `count(b)` (13%), never back (with a + b + c = 40, equally a dependence on count c). The same arm at 10K follows Amendment 1's one-third rule on 96% of these rows. That rule coincides with `count(a) + 1` on 16.8% of rows, so the 10K state's 0.187 share right is almost all one-third answers.

The pull is uneven. Where letters were removed, the 1K state answers `count(a) + 1` on about 90% of rows. Where two were added, it does so on a third, and LOW's own count of a is right on under a tenth. Its count of a copes with swapped text far better than LOW's, but not fully.

| state | `count(a) + 1` | strictly between it and `count(b)` | `count(b)` | elsewhere |
|---|---|---|---|---|
| slow, all spurious, 1K | 0.671 | 0.328 | 0.001 | 0 |
| same, seed 1 | 0.696 | 0.303 | 0.001 | 0 |
| slow, 80% spurious, 1K | 0.466 | 0.384 | 0.078 | 0.072 |
| slow, all spurious, 10K | 0.187 | 0.812 | 0.001 | 0 |
| LOW, before fine-tuning | 0.266 | 0.405 | 0.109 | 0.220 |

### What we learned from Amendment 4

On swapped rows where the two readings give different answers, the early slow all-spurious states answer one more than the count of a, on both seeds, and not the count of b. Their count of a copes with swapped text where LOW's does not. So the answer to Amendment 3's question is mostly the first reading: an existing count made stronger. It is not clean. The patch check clears its bar narrowly, and a third of their answers sit one step from `count(a) + 1` towards `count(b)`; in most of them that step is also the step back towards the pre-swap text, and on the 138 rows that separate the two, answers move only towards `count(b)` (13%), never back (exploration, not filed; with a + b + c = 40 this is equally a dependence on count c). Whether that is the shape of the rule of two counts the same arm reaches by 10K is an unfiled reading. The slow 80% state reads neither here.

### What it means for Stage C

The decision card's either/or, "they count b there, or they strengthened the count of a", now leans to the second for the slow all-spurious anchors, on rows with one or two letters swapped. This bears on the truth-set decision but does not settle it.

- Scoring the labelled states as filed rests on firmer ground for these two anchors; the slow 80% anchor is weaker.
- The design argument still fails: the anchors changed their count of a, by strengthening it, rather than leaving it alone.
- The natural-row recipe is still the way to get anchors nothing pushed to change, at the cost of a narrow label range.
- The third class gains a description, not a test: the late slow state follows the one-third rule on swapped rows too.

### How compelling is the evidence

- All 26 stop controls passed, and every natural patch share reproduced Stage A's exactly.
- Count b has a positive control; count a has none, so its "through" rests on the bars alone.
- The count-a patch share does not single out a strengthened count: on the same pairs LOW reads 0.575 and the 10K state 0.499, at or above the two seeds (0.505, 0.496). The strengthening rests on the behaviour share alone, 0.67 against LOW's 0.35 right.
- On off-label pairs the count-b patch share is 0.404 for LOW, which does not count b, so the two seeds' 0.19 is not evidence of a pull from b.
- Each share reads 1,000 rows or pairs. The count-a patch share clears its bar by 1.4 and 0.8 standard deviations.
- These states trained only on rows where count b is one more than count a; off-label rows are off that distribution.
- Only one or two swapped letters were read; training rows reach seven.

### Surprises

- LOW's `count(a) + 1` share is 0.27, not about zero: it lags the swap by about one letter, so where letters were removed it often lands on one more than the new count.
- The 1K state's count-a patch share fell from 0.72 on natural rows to 0.51, against about 0.7 expected. LOW and the 10K state read 0.50 to 0.58 on the same pairs, so the drop may belong to the pairs, not the state.
- The 10K state follows the one-third rule on swapped rows. Not filed.

### What next

1. **The Stage C truth-set decision**, with this input: the two all-spurious anchors keep "reuses" on swapped text, as a strengthened count of a (behaviour share; the patch share does not single it out).
2. **The natural-row re-run of the slow all-spurious arm**, if the user picks it. If it reads through count a with no step towards `count(b)`, the anchors no longer rest on swapped text.
3. **The same read at three to seven swapped letters**, at no cost. If the 1K state falls towards LOW, the strengthening is partial.

### Amendment 4 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| C0, W0 | controls; count-b control reads b in both parts | all checks; `reads b` | 26 of 26; Part 1 0.065, 0.748; Part 2 0.906, 0.766 | met; LOW's `count(a) + 1` share 0.266, above the expected 0 |
| P1 | slow all-spurious 1K reads through count a on swapped rows | Part 2 == `reads a` | 0.671, 0.505 through; 0.001, 0.191 neither | met, the 55% lean; narrow |
| P1s | its seed copy reads the same | void unless one-sided; else equal | `reads a`, `reads a` | met, the 85% lean |
| P2 | slow 80% 1K reads through count a | Part 2 == `reads a` | 0.466, 0.459, both just under the bars | missed; `neither`, the 35% second lean |
| Part 1 | patch on Stage A's swap-built rows, natural donors | described | `reads a`, `reads a`, `neither`; control `reads b`; LOW `both` | described; cannot separate, as filed |

### Amendment 4: not run

- A positive control for count a on swapped text; no saved model counts a there.
- Off-label rows with three or more letters swapped.
- A second seed of the slow 80% arm; more seeds of the slow all-spurious arm at 1K to 2K.
- Patches at other positions, or per subcomponent with a decomposition.
- What in swapped text the strengthened count copes with, and the route count b's pull takes.
- Whether the pairs of counts on which the 10K state follows the one-third rule here are new to it.

Provenance: `experiments/pcfg_anchors/results/amend4.json`, per-pair outputs `results/large/amend4/outputs/`, log `cluster/logs/commute-e55-amend4-44746.out`; job 44746, $0, 48 s, code at d62c43f; figure from `plot_amend4.py`.

## Amendment 5: do fine-tunes trained on natural text, where the old count already gives the answer, keep using it?

*2026-10-30. Code `experiments/pcfg_anchors/amend5.py`, predictions `results/predictions_amend5.json`, filed before the run and amended twice before the run; MATS cluster jobs 46022 and 46023, $0, about 7 min. Full tables, scoring and departures in `reports/analyses/2026-10-30_analysis_e55-amend5.md`.*

### The answer

Fine-tunes trained only on natural texts where one more than the count of a gives every answer do keep using the old count. All four seeds of the slow recipe answer through the count of a at 500, 1,000 and 2,000 steps, and never through a count of b. A fast copy does the same at first, then by 2,000 steps reads as neither. A positive control passed by 500 steps, and no state took a shortcut. This was close to guaranteed by how the rows were chosen, so its value is as ground truth for scoring methods, not as a discovery.

### What we were trying to test

Stage A left only two fine-tunes labelled "reuses the old count", and Amendments 3 and 4 showed both had been pushed to change that count: they trained on letter-swapped rows that LOW miscounts. Stage C needs anchors that nothing pushed.

So we sampled natural texts on which the count of b is already one more than the count of a, about one text in five. On every such row LOW's own count of a plus one is the label. Jain et al. build their spurious rows by editing texts instead; we depart from that because the edit breaks LOW's count. The cost is narrow labels: 69% of rows have label 14.

Three arms start from LOW, with Stage A's recipe otherwise:

- **slow natural-row**, at Stage A's slow rate, four seeds. Does the slow recipe keep the old count when nothing pushes it off?
- **fast natural-row**, at ten times the rate, one seed. Does the rate decide, not the rows?
- **prompt control**, two seeds: answer the count of a under `T a 40`, a pretrained prompt that asked for a terminal and never took a letter. LOW answers no count there. It checks that the checks read "through" for a fine-tune whose answers are the count of a. This arm is not in the paper. The plan's first choice, `C c 40`, was dropped after the code writer showed LOW already counts a there.

Each state is read by Stage A's two checks on 1,000 held-out natural texts where the count of b is not one more than the count of a. The **behaviour share** is the share answering one more than the count of a; for the control, the count of a itself. The **patch share** uses 1,000 pairs of texts that differ in the count of a but share the count of b. The residual at the answer position after block 5 is swapped from one text into the other, and the share is the fraction of answers that follow it. Both above Stage A's upper bars, 0.500 and 0.483, reads "through"; both under the lower bars reads "not through". A state **reuses** when the count of a is through and the count of b is not.

Jain et al.'s Def 2 calls this reuse when the fine-tuned map is $g \circ f_C$ for any $g$. We depart in two ways. We require $g$, here adding one, to work on held-out values too, which changes no reading here. And the causal checks and their "neither" class are this project's addition. A **shortcut** state answers 14 on rows labelled otherwise; its share is 0 for a correct answerer and 1 for a constant one.

States are saved at 500, 1,000 and 2,000 steps, where the paper reads only the end of training. The schedule is Stage A's 10,000-step one stopped at 2,000 steps, so every save sits in its warm-up. That is a recipe choice and a departure.

**Amended before the run.** Two dated amendments were made before any model was read. The lead lowered the control's lean from 85% to 65% and made the natural-row predictions void only if the control failed at every save. After the code review, the shortcut share was conditioned on the label (the evaluation rows are unseen training-distribution rows, Amendment 2's streams; no model trained on them). Unconditioned, a correct answerer would have scored 0.69 against the 0.6 bar, and any state through on neither count that still answered the training rows right would have been called a shortcut. Under the rule as first filed, the fast copy at 2K (accuracy 1.000, 0.6885 answered 14) would have read "shortcut"; the amendment, made before the run, makes it "neither". The file's definitions block still states the first rule; the amendments list governs, and the run used 0.5 on the label-conditioned share (amend5.json constants). The second amendment also moved P3's lean: neither or shortcut 40% to 45%, new or both 20% to new 15%. The scored "reuses" stays at 40%.

### What we expected

Each slow seed would reuse the old count at 1,000 steps, at 70%, and at 2,000 steps, at 60%. The fast copy was a toss-up. No shortcut state, at 75%. The control through the count of a at 1,000 steps, at 65%. On letter-swapped rows the natural-row states should answer like LOW, since they never saw swapped text.

### Do the natural-row fine-tunes run through the old count?

![Stage A's two count-a checks by save, for every Amendment 5 state](../experiments/pcfg_anchors/figures/amend5_saves.png)

**Reading it.** Yes for the slow recipe at every save, and for the fast one until 1,000 steps. The first point of every line is LOW before fine-tuning; "through" needs both panels in the green region. By 500 steps the four slow seeds answer one more than the count of a on about 0.98 of texts, where LOW answers it on none. Their patch share stays near 0.9 up to 1,000 steps, above Stage A's two earlier anchors. By 2,000 steps it drifts down to between 0.748 and 0.804. The fast copy weakens at each save and by 2,000 steps sits between the bars on both checks, so it reads neither. LOW's patch share is already 0.959, because it answers the count of a here; only the behaviour share separates it from the fine-tunes.

| state | behaviour, count a | patch, count a | patch, count b | class |
|---|---|---|---|---|
| LOW, before fine-tuning | 0.000 | 0.959 | 0.000 | neither |
| slow natural-row, 4 seeds, 1K | 0.976 to 0.981 | 0.898 to 0.911 | 0.007 to 0.010 | reuses |
| slow natural-row, 4 seeds, 2K | 0.940 to 0.959 | 0.748 to 0.804 | 0.015 to 0.044 | reuses |
| fast natural-row, 1K / 2K | 0.843 / 0.407 | 0.627 / 0.373 | 0.190 / 0.367 | reuses / neither |
| Stage A slow all-spurious, 1K | 0.879 | 0.715 | 0.104 | reuses |

### Did any state take a shortcut, and does the positive control pass?

No shortcut, and the control passes. Every natural-row state answers 14 on none of the unseen training-distribution rows labelled otherwise, where LOW does on 0.248 of them. Every state, including the fast copy at 2,000 steps, answers all 4,000 unseen training-distribution rows (Amendment 2's streams; no model trained on them) right. So the fast copy's drift is a change of rule away from the training distribution, not a constant answer on it.

Under the T prompt both control seeds answer the count of a on every held-out text at every save, against none for LOW. Their patch share is about 0.9. One caveat: LOW's own patch share there is 0.655, already above the bar, because the patch at block 5 carries whatever answer the model gives. So the control's pass rests on its behaviour share, which went from 0 to 1.

On Amendment 4's letter-swapped rows the slow seeds answer one more than the count of a on about 0.30 of rows. LOW does on 0.27, and Stage A's swap-trained anchor on 0.67. So, as expected, the natural-row states answer swapped text like LOW.

### What we learned from Amendment 5

A fine-tune whose training rows are all answered by the old count plus one keeps using that count for at least 2,000 slow steps, in every seed. In the one fast seed, at ten times the rate, the state reads neither by 2,000 steps (count-a behaviour 0.407) while still answering every training row right. So the reading depends on the rate and on when it is read; the rows were not varied in this amendment.

### What it means for Stage C

- **The 'reuses' side grows from two states to sixteen** (seventeen counting Stage A's seed copy, which is outside the eleven labelled states), all trained on rows LOW's own count answers. Which saves enter the truth set was not filed; the Stage C plan fixes it before scoring. They come from five runs, four seeds of one recipe and one fast copy. The fast copy sees the same rows in the same order as slow seed 0, so it is a paired comparison of learning rate, not an independent run: four independent runs, seven with Stage A's three.
- **Option A**, scoring the labelled states, needs its pass bar set again: the set now leans towards "reuses", sixteen against nine "new".
- **Option B**, the unscored third class, holds the three all-spurious states at 10,000 steps that answer a rule of two counts. The fast natural-row state at 2,000 steps resembles them on the two count-a checks (count-b patch 0.367 vs 0.43; not filed), but joins only if a row-by-row read shows the same rule.
- The prompt control shows the checks read "through" for answers that are the count of a, and is not part of the truth set.

### How compelling is the evidence

- "Reuses" was close to guaranteed: adding one to an answer LOW already gives is the easiest fit, and the loss fell near zero by 500 steps.
- The prompt control is a control, not a test. It cannot show the prompt was routed into LOW's counter rather than a new one, since the patch share was above the bar before training; its pass rests on behaviour.
- **The natural-row readings rest on behaviour too.** LOW's patch share for count a at `C b 40` is 0.959 before training, above the bar; the patch adds only that the answer is still carried at block 5. The block-5 patch passes any answer that differs between the two texts of a pair, as LOW's terminal answer under the T prompt shows (0.655).
- The reading depends on when it is taken: the fast copy changed between 1,000 and 2,000 steps, and nothing was read later.
- Every save sits in the warm-up of a cut schedule.
- One base model, one grammar; four slow seeds, one fast, two control.

### Surprises

- The control passed by 500 steps, faster than its lowered lean expected.
- The fast copy's two count-a shares at 2,000 steps (not its count-b patch, 0.367 vs 0.43) sit within 0.02 of Stage A's slow all-spurious arm at 10,000 steps. Not filed.
- No filed prediction was reversed.

### What next

1. **Stage C, option A with B,** once the pass bar is set for the new set.
2. **A row-by-row read of the fast natural-row state at 2,000 steps,** $0. If it follows the rule of two counts, that rule is not an effect of swapped text.
3. **The slow natural-row recipe read to 10,000 steps,** $0. If it leaves the old count, the anchors' label belongs to early training only.

### Departures from the literature

- Bounded $g$: $g$ must work on held-out values; the paper accepts any $g$.
- Natural rows that already carry the correlation, instead of the paper's edited rows.
- The prompt control, which the paper does not have.
- The behaviour and patch checks, their bars and the "neither" class, added to Def 2 by this project.
- Saves at 500 to 2,000 steps inside the warm-up of a cut schedule, where the paper reads the end of training.- **Evidence.** The paper never tests Def 2 directly. Its evidence is pruning, probing and reverse fine-tuning (§5). None is run here; the behaviour and patch checks replace them.
- **Unmatched label histogram.** The paper matches the class histograms of spurious and clean rows (Fig. 16). Ours are not matched: 69% of training rows carry label 14.
- **The shortcut check** is this project's. The paper's guard against a constant answer is a class-histogram check (Fig. 59).
- **Read position.** The patch is at `=`, the position that predicts the answer. The paper reads probes at the answer token.
- **Learning rates.** 1e-5 and 1e-4 are the paper's medium and large rates. Its small rate, 1e-6, where it reports wrappers at every test-time spurious rate, is not run.
- **Inherited from Stage A:** weight decay 0.1 from minGPT's defaults (not stated in the paper), pretraining details, and evaluation on qualifying texts rather than the paper's test sets; see the Stage 1 plan's recipe table.


### Amendment 5 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| W0 | LOW reproduces its references, gives every label, does not count a under the T prompt | within 0.01; ≥ 0.99; ≤ 0.05 | 0.99975, 0.960; 1.000; 0.000 | met, 95% lean |
| P1 | each slow natural-row seed reuses at 1K | class == reuses | 4 of 4 | met, 70% lean |
| P2 | the same at 2K | class == reuses | 4 of 4 | met, 60% lean |
| P3 | the fast copy reuses at 1K | class == reuses | reuses; count b neither, 0.190 | met, 40% lean |
| P4 | no shortcut state at 1K | no state at 1K classed shortcut (share ≥ 0.5 on a state through on neither count) | 0.000 everywhere | met, but untestable: no state at 1K was through on neither count; the share 0.000 at every save is the informative reading (described) |
| P5 | the prompt control is through count a at 1K | label == through | 1.000, 0.911; seed 1 1.000, 0.914 | met, 65% amended lean |
| D1 | natural-row states answer swapped text like LOW | described | 0.30 vs LOW 0.27 | as expected |

### Amendment 5: not run

- Any natural-row state past 2,000 steps; more fast seeds; natural rows with wider labels.
- Row-by-row reads of any state here.
- Patches at other positions or blocks, or per subcomponent with a decomposition.
- Stage B's bounded-$g$ readings; Stage C's instruments; VPD.
- Stage A's "new" positive control (N1) re-read through this code.
- The paper's small rate, 1e-6.
- Pruning, probing and reverse fine-tuning on these states (Stage C1).

Provenance: `experiments/pcfg_anchors/results/amend5.json` at 82f6726, states in `results/large/amend5/`, logs `cluster/logs/commute-e55-amend5c-46022.out` and `commute-e55-amend5-46023.out`; jobs 46022, 38 s, and 46023, 6 min 21 s, $0, code at 7b5ef02; figure from `plot_amend5.py`.

## Stage C1: do pruning, reverse fine-tuning or steering tell "reuses the old count" from "new count" better than the training recipe does?

*2026-10-30. Code `experiments/pcfg_anchors/c1.py`, `amend5b.py`, `src/commute/revival.py`; predictions `results/predictions_c1.json` and `predictions_amend5b.json`, filed before any run; MATS cluster jobs 46105, 46106, 46107 and 46121, $0, about 50 min. Every number, the per-state table and the scoring in `reports/analyses/2026-10-30_analysis_e55-c1.md`.*

### The answer

No instrument passed the filed test on every state. Pruning and steering agreed with the ground truth on about four states in five, above chance, but a labeller that only reads the fine-tune's learning rate came within one state of each, short of the filed margin of two. Reverse fine-tuning called almost every state "reuses" and missed the chance bar. On one vote per training run, pruning and steering passed. The surprise is not filed: each instrument's raw reading separates the two sides with no overlap, so the filed cut-offs were too lenient.

### What we were trying to test

Stage A and Amendment 5 gave a ground truth with no decomposition: 15 fine-tuned states that answer through LOW's old count of a, and 11 that do not. Stage C1 asks whether three instruments that also need no decomposition recover those labels:

- **Pruning revival**, Jain et al.'s Alg. 3: zero the 5 or 15 MLP units whose removal should most lower the count-a loss, and compare count-a accuracy with 20 random removals. "Existing" when the gain is at least 3 standard deviations.
- **Reverse fine-tuning**: train the state back on the count of a at a tenth of its own rate until 95% accuracy, against NEVER, a model that never counted a, trained the same way. "Existing" at half of NEVER's steps or fewer.
- **Steering**, E52's mean-shift baseline: add the mean difference of hidden states after one block to LOW, and read the share $S$ of the fine-tune's gain in log-probability of its own answer that this recovers. $S = 0$ is no effect and $S = 1$ the whole fine-tune. "Existing" when $S \ge 0.5$.

The truth set is confounded with the recipe: every "reuses" state is an early save, and most new states are fast or long fine-tunes. So two labellers that read only the recipe, "fine-tuned for at most 1,000 steps" and "fine-tune rate at most 1e-5", are the reference. An instrument passes when it clears the binomial bar and beats the better of them by two states. Beside it, a per-run score keeps one vote per training run, with a margin of one.

Amendment 5b first added three more fast natural-row seeds, so that the "reuses" side has states at the fast rate too.

### What we expected

Each instrument was expected to miss: pruning and reverse fine-tuning at 20% to pass, because Jain et al. find revival after every fast fine-tune, steering at 40%. The controls were expected to hold.

### Does any instrument beat the recipe-only labellers?

![Agreement with the truth per instrument and nuisance labeller](../experiments/pcfg_anchors/figures/c1_scores.png)

**Reading it.** No, not on every state. The bars are each method's share of states labelled as the truth labels them; the black tick is the binomial bar and the red tick what the instrument needed. Pruning agreed on 20 of 26 states and steering on 21 of 27, both above their bars. The labeller that reads only the learning rate agreed on 19 and 20 of the same states. So each instrument fell one state short of the margin of two. Reverse fine-tuning agreed on 17 of 26, under its bar of 18. On one vote per run, pruning at 14 of 17 and steering at 16 of 18 pass.

The controls held. Importance pruning on LOW removed 229 and 816 of 4,000 correct answers, where random removals removed at most 3. The filed bar was about 3 rows, so this control shows only that the score finds important units. NEVER needed 2,300 steps at 1e-5 and never finished at 1e-6.

| instrument | every state | needed | one vote per run | needed |
|---|---|---|---|---|
| pruning | 20 of 26 | 21 | 14 of 17 | 14 |
| reverse fine-tuning | 17 of 26 | 21 | 12 of 17 | 14 |
| steering | 21 of 27 | 22 | 16 of 18 | 14 |
| learning-rate labeller | 19 of 26 | reference | 10 of 17 | reference |
| step labeller | 18 of 26 | reference | 13 of 17 | reference |

### Why do the label rules miss?

![Pruning gain over random draws per state](../experiments/pcfg_anchors/figures/c1_prune.png)

**Reading it.** Pruning revives the count of a by 0.35 to 0.75 in every "reuses" state, and by at most 0.08 in new ones. The filed rule needs only 3 standard deviations, 0.02 to 0.03 here, so six new states with gains of 0.05 to 0.08 read "existing". The size of the revival separates the two sides; the rule's cut-off does not.

![Reverse fine-tuning steps per state](../experiments/pcfg_anchors/figures/c1_reverse.png)

**Reading it.** Reverse fine-tuning brings the count of a back quickly almost everywhere, so its label says "existing" for 24 of 26 states. At 1e-6 NEVER never finished, which put the cut-off at 5,000 steps, and every state came back within 3,300. The prediction expected the slow states to be slow; they were not. Within each rate, though, every "reuses" state is faster than every new one.

![Steering gain share per state against fine-tuning steps](../experiments/pcfg_anchors/figures/c1_steer.png)

**Reading it.** One mean shift recovers 0.84 to 0.98 of every "reuses" state's gain, and 0.56 to 0.67 of the new states at 1,000 steps. Those clear the filed line of 0.5, so at 1,000 steps steering labels like the step labeller. R, fine-tuned from a model that already counts b, has a gain of 0.002 against 6 to 11 elsewhere. Its 10,000-step vote, a "new" against a "reuses" truth, cannot be read. Dropping it after the fact, steering would agree on 21 of 26 and meet the requirement of 21 exactly; the filed miss stands.

All three raw readings separate the 26 states with no overlap. The gaps were found by looking, so a cut-off placed in them is fitted, and at the fast rate they may also separate Amendment 5's rows from Stage A's. Class B spreads across every gap.

### Amendment 5b: do three more fast natural-row seeds reuse the old count at 1,000 steps?

![Count-a checks by save for the fast natural-row seeds](../experiments/pcfg_anchors/figures/amend5b_saves.png)

**Reading it.** Yes, all three, and all three read neither by 2,000 steps, as seed 0 did. At 1,000 steps the behaviour shares are 0.66 to 0.79 and the patch shares 0.51 to 0.60, above the bars of 0.500 and 0.483. Seed 3's patch share clears its bar by 0.027, under two binomial standard deviations. The count-b patch share sits between the bars at 1,000 steps, so these states reuse under the filed rule but not the plan's literal wording, as seed 0. LOW's controls reproduced exactly and no state took a shortcut. The three 1,000-step states joined the "reuses" side of C1, the 2,000-step ones class B.

### What we learned from Stage C1

None of the three cheap instruments labels reuse better than reading the learning rate does, under the filed rules. Their raw readings, though, order the states the way the truth does, so the instruments carry the signal and the filed cut-offs threw it away. For `GOALS.md`, a decomposition scored against this truth must clear the recipe-only labellers, and no method without one has done so yet.

### How compelling is the evidence

- 26 scored states from 17 runs; saves of one run, and runs sharing rows, are not independent.
- The truth is confounded with the recipe, which is why the recipe-only labellers score 18 and 19 of 26.
- The pruning control's bar is about 3 rows; reverse fine-tuning's comparison at 1e-6 is with a cap; steering has no positive control, and R's votes are unreadable.
- "Reuses" states answer one more than the count of a, a wrapper Jain et al. say pruning and retraining should find, so agreement here is easy.
- One base model, one grammar, rates 1e-4 and 1e-5.

### Surprises

- Every raw reading separates the truth with no overlap, while every label rule errs toward "existing". Not filed.
- Slow new states came back under reverse fine-tuning in 1,800 to 3,300 steps, where the prediction expected slow revival or the cap.
- Alg. 3's own direction lowered LOW's count of a, by 0.04 to 0.06.
- The steering miss rests on one unreadable vote.
- No filed prediction was reversed: all three leaned toward a miss.

### What next

1. **Score VPD against the same truth and the same recipe-only labellers**, after the user's go-ahead. If it cannot beat the learning-rate labeller either, the truth set is the bottleneck.
2. **A filed rerun with cut-offs set on other states**, $0. If the gaps hold on states they were not fitted to, the instruments read reuse.
3. **A truth set without the recipe confound**: "reuses" states read late, new ones early.

### Departures from the literature

- Reverse fine-tuning with Stage A's AdamW recipe at a tenth of the fine-tune rate, where the paper uses its Tracr SGD recipe; a scratch model that never counted a, where the paper's counted the new operand; steps to 95% at half of scratch's as the label.
- Pruning of MLP hidden units only, K of 5 and 15, against 20 random removals with a 3 standard deviation rule; the paper plots accuracy curves with no baseline.
- A pruning control, steering, the nuisance labellers, the per-run score and the flag on gains under 0.01: none is in the paper.
- States read at 1,000 or 2,000 steps, inside the warm-up, where the paper reads the end of training.
- Bounded $g$ and the behaviour and patch checks, as in Stage A and Amendment 5.

### Stage C1 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| C0 | the instruments' controls hold | pruning drop ≥ 3 sd on LOW; scratch > 1,000 steps | 229 and 816 rows vs 3; 2,300 and cap | met; pruning control near-vacuous |
| PC1 | pruning agrees above chance | ≥ bar and ≥ better recipe labeller + 2 | 20 of 26, needs 21 | not met, 20% lean; per run passes |
| PC2 | reverse fine-tuning agrees above chance | same | 17 of 26, bar 18 | not met, 20% lean; per run fails |
| PC3 | steering agrees above chance | same, with R | 21 of 27, needs 22 | not met, 40% lean; per run passes |
| 5b P1 | each fast natural-row seed 1-3 reuses at 1K | class == reuses | 3 of 3 | met, 50% lean |

### Stage C1: not run

- VPD on these states; Stage B.
- Cut-offs set on states outside the truth set.
- Other reverse rates, the paper's scratch baseline, attention-unit pruning, input-dependent steering.
- A truth set without the recipe confound.

Provenance: `results/c1.json` at 3f7a972, `results/amend5b.json` at dcbd5f6; jobs 46105, 46106, 46107 and 46121, $0; code reviewed at 8de9324, `reports/reviews/2026-10-30_review_e55-c1-code.md`; figures from `plot_c1.py`.

## Stage C1b: with cut-offs set on other seeds, do pruning, reverse fine-tuning and steering tell reuse from a new count?

*2026-10-31. Code `experiments/pcfg_anchors/c1b.py`, `src/commute/revival.py`; predictions `results/predictions_c1b.json`, filed before any run; cut-offs `results/c1b_cutoffs.json`, committed before any scored state was read; MATS cluster jobs 46330 to 46499, $0, about 2.5 h. Every number, the per-state table and the scoring in `reports/analyses/2026-10-31_analysis_e55-c1b.md`.*

### The answer

All three cheap methods passed the filed test, on every scored state and on one vote per run. Each cut-off, set on two held-out seeds, put all 21 scored states on the right side, with room to spare against each method's noise. The test was easier than it looks. Every state performs one of two fixed tasks perfectly, so anything that tells the tasks apart scores full marks. The answer-only probe does, and so does the size of the weight change within a recipe. The benchmark has no headroom left to rank VPD above these methods.

### What we were trying to test

Stage C1's truth set let the recipe almost match the labels, and its cut-offs were guessed. C1b fixes both. Every state is LOW fine-tuned on natural `C b 40` text, never on swap-built rows, with one of two tasks:

- **Task P, plus one**: answer one more than the count of a on every row. The count of b fits only 18% of rows, so a state that performs P must be computing the count of a. By construction it **reuses** the old count.
- **Task N**: answer the count of b, which does not depend on the count of a. The state has a **new** count.

Both tasks run at learning rates 1e-4 and 1e-5, saved at 1,000 and 10,000 steps, with five seeds. Seeds 0 and 1 set each method's cut-off, at the midpoint of the gap between the classes; seeds 2 to 4 are scored. Task N at 1e-5 had not learned by 1,000 steps, so 21 states are scored. The methods are C1's: pruning revival, reverse fine-tuning, now at one rate of 1e-5 for every state, and steering's gain share $S$. A fourth, a probe on LOW's activations that predicts the state's own answers, reads nothing but the answers; it is reported as a check on the truth set and never scored.

A method passes when it clears the binomial bar and beats the best recipe-only labeller by two states. The recipe-only labellers read the learning rate, the step count and the update size $\|W - W_0\|_F / \|W_0\|_F$, each at its best threshold in hindsight.

### What we expected

The controls and the truth set were expected to hold, at 90% and 85%. Pruning and steering were expected to pass at 60%, reverse fine-tuning at 45%. The filing gave 40% to the update size separating the tasks well enough to raise the bar.

### Does each cut-off, set on other seeds, split the scored states?

![Each instrument's reading per state, grouped by recipe](../experiments/pcfg_anchors/figures/c1b_readings.png)

**Reading it.** Yes, for all three, and with a wide gap. Each panel groups the states by recipe, task P in green on the left and task N in blue on the right. Hollow points are the calibration seeds that set the red cut-off; filled points are the scored seeds. Every filled green point sits on the "reuses" side and every filled blue point on the "new" side. Steering sits closest to its line, about 0.09 away on both sides, three times its noise between the two halves of its rows. Pruning's nearest states are 0.38 above and 0.14 below its line. Reverse fine-tuning's are 325 steps either side, counted in units of 50. Against the references, random removal revives nothing, while every P state revives 0.60 to 0.92 of count a. Scratch needs 2,300 steps to relearn the count of a, while P states need 250 or 300.

### Does any method beat the recipe, and what else scores as well?

![Agreement with the truth per method and per labeller](../experiments/pcfg_anchors/figures/c1b_scores.png)

**Reading it.** The three methods beat the filed recipe labellers easily, but they do not beat two simpler signals. Each agrees with the truth on 21 of 21 states against a requirement of 17, and on 12 of 12 runs against 10. The best filed labeller, update size at one threshold, reaches 15, so the methods clear it by six states. But the probe, which reads only the answers, also reaches 21 of 21. So does a post hoc labeller: update size with one cut-off per recipe, each set on the calibration seeds by the methods' own rule. Neither is scored, and both tie the three methods.

### Why does the update size read the task within a recipe?

![Update size per state, grouped by recipe](../experiments/pcfg_anchors/figures/c1b_update_size.png)

**Reading it.** Learning the new count moves the weights further than adding one, at every recipe. Within each group every blue N state sits above every green P state, by a factor of 1.15 to 1.45. Across recipes the sizes span a factor of 30, so no single threshold separates the tasks; the grey line, the best one, gets 15 of 21. The plan named update size a nuisance. It may instead be a real signal, since a new count may simply need a larger change than a shift of an old one. Either way it needs only the weights, and a decomposition would have to beat it.

### What we learned from Stage C1b

With cut-offs set on held-out seeds, pruning revival, reverse fine-tuning and steering each label every scored state right. C1's misses came from its guessed cut-offs: with the new ones, C1's own readings sort its states too, described only. But on this set the label is the same thing as which of two tasks the state performs. Any reader of the task, such as the answer-only probe or the update size within a recipe, ties the methods. So this truth set can show that a decomposition is as good as these methods, but not that it is better.

### How compelling is the evidence

- **Two tasks, each done perfectly.** Every P state answers one more than the count of a on every row, and every N state the count of b. The probe reads the same value on all 20 P states because their answers are identical. A method that reads only which task was trained cannot be told apart from one that reads the mechanism.
- **Wide margins.** The closest method, steering, sits about three noise widths from its cut-off; every scored reading lies within its calibration seeds' range.
- **No swap-built rows** in training, evaluation, steering or the probe.
- **Small and partly paired.** 21 states from 12 runs; P and N runs with the same seed share their texts. One model, one grammar.

### Surprises

- The update size separates the two tasks perfectly within every recipe. The filing did not consider a within-recipe reading.
- The filing's lean called the update size reaching 15 and the "no method can pass" stop the same event. They are not: at 15 the requirement is 17, and the stop did not fire.
- No filed prediction was reversed.

### What next

1. **Decide what Stage C2 can show before spending on it.** On this set VPD can at best tie, as the probe and the update size already do. A tie would say VPD reads the task, not that it reads reuse better.
2. **A truth set where the answers do not give the label away**: pairs with the same answers and different mechanisms, such as task P fine-tuned from NEVER, which must build a count of a, against task P from LOW. Needs `/design` first.
3. **Add the update size within a recipe to the nuisance labellers**, or argue in a decision row that it is a legitimate reading.

### Departures from the literature

- Task P is not in Jain et al.; it removes their wrapper setting's competing answer so the label holds at 10,000 steps.
- Pruning of MLP units only, 5 or 15, against 20 random removals, with a calibrated cut-off; the paper plots curves.
- Reverse fine-tuning at one AdamW rate for every state, capped at 5,000 steps, against NEVER; the paper uses its SGD recipe below the fine-tune's rate and a scratch model pretrained on the target.
- Steering adds the mean difference between the state and LOW to LOW and scores a share of the gain; CAA adds a contrastive mean difference to the same model and scores behaviour.
- The probe predicts the state's answers from LOW's activations, with shuffled labels for Hewitt and Liang's control task.
- Calibration, the recipe labellers, the margin, the binomial bar and the per-run score are this project's.

### Stage C1b scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| C0 | controls hold | `rules.controls_stops` | LOW 0.99975 / 0.960; scratch 2,300; identity-prompt S 0.94 / 0.93 | met, 90% lean |
| C1 | the truth forms as built; recipe cannot read it | `rules.train_stops` | 0 disagreements; n 21; ceiling 12 / 12 / 15, needs 17 | met, 85% lean; update size branch happened |
| PC1 | pruning beats chance and the recipe by 2 | cut-off, label, score | 21 of 21, needs 17; per run 12 of 12 | met, 60% lean |
| PC2 | reverse fine-tuning, same | same | 21 of 21; per run 12 of 12 | met, 45% lean |
| PC3 | steering, same | same | 21 of 21; per run 12 of 12 | met, 60% lean |
| PC4 | probe agrees with the task, reference only | not a pass | 21 of 21 | as expected |

### Stage C1b: not run

- VPD on these states, Stage C2.
- States with matched answers and different mechanisms.
- A second reuse task with a wrapper other than plus one.
- Attention-unit pruning, other reverse rates, CMAP and dictionary instruments.

Provenance: `results/c1b.json` at 50bd0cd, cut-offs at d361649; jobs 46330, 46346, 46381, 46399, 46443, 46459, 46476 and 46499, $0; code review `reports/reviews/2026-10-30_review_e55-c1b-code.md`; figures from `plot_c1b.py`.